# Score Test — Optimized Playground Notebook
Converted from `score_test.py` with runtime optimizations:
- Parallel temporal window scoring via `ThreadPoolExecutor`
- Single-pass temporal merge (collect → concat → merge once)
- Vectorized regime labelling (no Python-level list comp)
- Cache-delete toggle
- Pre-filtered copies only where mutation is needed

In [1]:
import pandas as pd
import numpy as np
import os, json, datetime as dt, time
from typing import cast
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys
sys.path.append('..')

import data_lib.config as config
import data_lib.stocks.stocks_config as sc
from data_lib.data_fetch.get_data import get_test_data, get_g1_distance
from data_lib.compute import process
from data_lib.feature.active_base_features import compute_active_base_features
from data_lib.geometry.geometric_features import batch_compute_geometry, calculate_adaptive_h
from steps.step3_simpulate import run_declines_simulation
from data_lib.data_fetch.get_ops_data import build_partner_ops_vector, get_active_base
from data_lib.stocks.gatekeeper import run_gates
from data_lib.feature.ops_features import compute_operational_score
from data_lib.feature.composite import compute_composite


/Users/Rohanchoudhary/Desktop/projs/genie_stocks/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ════════════════════════════════════════════
# USER FLAGS
# ════════════════════════════════════════════
DELETE_CACHE   = True   # Set True to nuke existing scored_df cache
RUN_SIMULATE   = False   # Set True to run declines simulation
MAX_WORKERS    = 4       # Parallel workers for temporal window scoring


In [3]:
BASE_DIR    = os.path.dirname(os.path.abspath("__file__"))
PARENT_DIR  = os.path.dirname(BASE_DIR)
ARTIFACTS_DIR = os.path.join(PARENT_DIR, "artifacts")
REPORTS_DIR   = os.path.join(PARENT_DIR, "reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

cache_scored_csv = os.path.join(REPORTS_DIR, "scored_df.csv")
cache_scored_h5  = os.path.join(REPORTS_DIR, "scored_df.h5")

if DELETE_CACHE:
    for p in (cache_scored_csv, cache_scored_h5):
        if os.path.exists(p):
            os.remove(p)
            print(f"Deleted cache: {p}")


In [4]:
t0 = time.perf_counter()

poly_path  = os.path.join(ARTIFACTS_DIR, "poly_stats_final.h5")
bound_path = os.path.join(ARTIFACTS_DIR, "partner_cluster_boundaries.h5")
train_path = os.path.join(ARTIFACTS_DIR, "train_data.h5")

df_poly  = cast(pd.DataFrame, pd.read_hdf(poly_path, "df"))
df_bound = cast(pd.DataFrame, pd.read_hdf(bound_path, "df"))
df_train = cast(pd.DataFrame, pd.read_hdf(train_path, "df"))

if config.DEFINITE_DECISIONS == 1:
    df_train = df_train.loc[
        df_train["final_decision"].isin(["DECLINED", "INSTALLED"])
    ].copy()

df_train["h"] = np.where(
    df_train["field_weight"] >= 0, config.H_INSTALL, config.H_DECLINE
)

if config.USE_ADAPTIVE_H:
    print(f"Adaptive H  k={config.ADAPTIVE_H_NEIGHBOR_K}, "
          f"min={config.ADAPTIVE_H_MIN}m, max={config.ADAPTIVE_H_MAX}m")
    df_train = calculate_adaptive_h(df_train)

df_train_full = df_train  # keep ref; .copy() avoided — no mutation below
print(f"Artifacts loaded  ({time.perf_counter()-t0:.1f}s)")


Adaptive H  k=3, min=20.0m, max=150.0m
Artifacts loaded  (8.0s)


In [5]:
t0 = time.perf_counter()
print(f"Test window: {config.TEST_START_DATE} -> {config.TEST_END_DATE}")
df_test = get_test_data(config.TEST_START_DATE, config.TEST_END_DATE)
assert not df_test.empty, "No test data returned"

before = len(df_test)
df_test = (
    df_test
    .groupby(["mobile", "latitude", "longitude"], as_index=False)
    .agg(
        partner_id=("partner_id", "first"),
        nmbr_partners=("partner_id", "count"),
        decision_time=("decision_time", "min"),
        installed_decision=("installed_decision", "max"),
        installed_time=("installed_time", "max"),
    )
)
print(f"Aggregated: {before} -> {len(df_test)} unique mobiles  "
      f"({time.perf_counter()-t0:.1f}s)")


Test window: 2025-10-20 -> 2025-11-09
Fetching TEST data (2025-10-20 to 2025-11-09)...
[WiomData] snowflake_select_start: select partner_id, mobile, first_notified_time,
               case when lco_account_installed = par
Standardization Cutoff (p40): 288.63 minutes
STANDARDISED DECISIONS (TRAIN):
final_decision
DECLINED         12334
INSTALLED        10477
INDETERMINATE     5635
HANGING           2697
Name: count, dtype: int64
Aggregated: 31143 -> 17574 unique mobiles  (6.3s)


In [6]:
t0 = time.perf_counter()
df_ops_train = build_partner_ops_vector(config.TRAIN_START_DATE, config.TRAIN_END_DATE)
df_ops_test  = build_partner_ops_vector(config.TEST_START_DATE, config.TEST_END_DATE)
print(f"OPS vectors: train {df_ops_train.shape}, test {df_ops_test.shape}")

df_test, df_ops_train = run_gates(df_test, df_train, df_ops_train)
if not df_ops_train.empty:
    df_ops_train = compute_operational_score(df_ops_train)
print(f"Gates done  ({time.perf_counter()-t0:.1f}s)")


[OPS VECTOR] Pulling lead lifecycle...
[WiomData] snowflake_select_start: WITH interest AS (
        SELECT
            mobile,
            TRY_PARSE_JSON(data):partner_id::S
[OPS VECTOR] Pulling ticket tasks...
[WiomData] snowflake_select_start: WITH assign_events AS (
        SELECT
            account_id                     AS long_customer_i
[OPS VECTOR] Pulling slot confirmation...
[WiomData] snowflake_select_start: WITH slots AS (
        SELECT
            mobile,
            TRY_PARSE_JSON(data):partner_id::STRI
[OPS VECTOR] Pulling partner performance (windows=[30, 60, 365])...
[WiomData] snowflake_select_start: SELECT
        partner_id,
        
        COUNT(DISTINCT CASE WHEN first_notified_time >= DATEADD(
[OPS VECTOR] Pulling expected daily slots...
[WiomData] snowflake_select_start: WITH daily AS (
        SELECT
            TRY_PARSE_JSON(data):lco_account_id::STRING AS partner_id
[OPS VECTOR] Computing shock flags...
[WiomData] snowflake_select_start: SELECT
        p

In [7]:
t0 = time.perf_counter()
print("Computing geometric features...")
df_train_geom = df_train
if config.GEOM_INSTALL_FILTER == 1:
    df_train_geom = df_train.loc[df_train["installed_decision"] == 1]

geom_features = batch_compute_geometry(
    df_test, df_train_geom,
    radius_m=int(config.GEOM_SEARCH_RADIUS_M),
    reference_date=config.TEST_START_DATE,
)
df_test = pd.concat(
    [df_test.reset_index(drop=True), geom_features], axis=1
)
print(f"Geometric features done  ({time.perf_counter()-t0:.1f}s)")


Computing geometric features...
[GEOMETRY] Computing temporal geometry for windows: [30, 60, 365]
[GEOMETRY]   30d: 7139/17574 leads with valid geometry
[GEOMETRY]   60d: 10356/17574 leads with valid geometry
[GEOMETRY]   365d: 15150/17574 leads with valid geometry
Geometric features done  (15.0s)


In [8]:
# Vectorised regime labelling — replaces per-row Python loop
d = (df_test["dense_score"]  >= config.GEOM_DENSE_THRESHOLD).astype(int)
g = (df_test["gully_score"]  >= config.GEOM_GULLY_THRESHOLD).astype(int)
s = (df_test["sparse_score"] >= config.GEOM_SPARSE_THRESHOLD).astype(int)

lookup = {
    (0,0,0): "mixed",
    (1,0,0): "dense",  (0,1,0): "gully",  (0,0,1): "sparse",
    (1,1,0): "dense+gully", (1,0,1): "dense+sparse", (0,1,1): "gully+sparse",
    (1,1,1): "dense+gully+sparse",
}
keys = list(zip(d, g, s))
df_test["geometry_regime"] = [lookup[k] for k in keys]


In [ ]:
# t0 = time.perf_counter()
# df_active = get_active_base(start_dt = config.TRAIN_START_DATE, end_dt=config.TRAIN_END_DATE)
# print(df_active.columns)
# test_mobiles = set(df_test["mobile"].astype(str))
# df_active = df_active[~df_active["mobile"].astype(str).isin(test_mobiles)]
# print(f"Active base: {len(df_active)} customer points")

# ab_features = compute_active_base_features(
#     df_test, df_active, radii_m=[50, 100, 200, 500]
# )
# df_test = df_test.merge(ab_features, on="mobile", how="left")
# print(f"  ab_count_100m median={df_test['ab_count_100m'].median():.0f}")
# print(f"  ab_nearest_dist_m median={df_test['ab_nearest_dist_m'].median():.0f}")
# print(f"Active base done  ({time.perf_counter()-t0:.1f}s)")


[WiomData] snowflake_select_start: SELECT
        a.partner_id,
        a.long_customer_account_id,
        a.mobile,
        a.latitud
RangeIndex(start=0, stop=0, step=1)


KeyError: 'mobile'

In [10]:
t0 = time.perf_counter()
if os.path.exists(cache_scored_h5):
    print("Loading scored_df from cache...")
    scored_df = pd.read_hdf(cache_scored_h5, "df")
else:
    print("Cache miss — running process()...")
    scored_df = process(
        df_train, df_test, df_poly, df_bound,
        lambda_decay=config.LAMBDA_DECAY,
        max_radius_m=int(config.MIN_DIST_CUTOFF_M),
    )
    scored_df.to_csv(cache_scored_csv, index=False)
    cat_cols = scored_df.select_dtypes(include=["category"]).columns
    if len(cat_cols):
        scored_df[cat_cols] = scored_df[cat_cols].astype(str)
    scored_df.to_hdf(cache_scored_h5, key="df", mode="w")

assert scored_df is not None and not scored_df.empty, "Scoring empty"
required = ["parent_color", "parent_installs",
            "predicted_field_hex", "contested_field"]
missing = [c for c in required if c not in scored_df.columns]
assert not missing, f"Missing columns: {missing}"
print(f"Main scoring done  ({time.perf_counter()-t0:.1f}s)")


Cache miss — running process()...


100%|██████████| 17572/17572 [00:43<00:00, 405.83it/s]


[HEX CONSENSUS] Temporal windows available: [30, 60, 365]
[HOP FEATURES] Computing 3-hop neighbor SE aggregates (+ temporal)...
[HOP FEATURES] 123493 hex rows with hop features
[ALL-HEX FIELD] Starting combined field computation across all overlapping hexagons...
[ALL-HEX FIELD] Targets × all hexagons sjoin: 137340 rows
[ALL-HEX FIELD] Sources × filtered hexagons sjoin: 5531996 rows
[ALL-HEX FIELD] After partner match filter: 432391 source rows
[ALL-HEX FIELD] Processing 41119 hex groups...


All-hex combined field: 100%|██████████| 41119/41119 [08:25<00:00, 81.42it/s]


[ALL-HEX FIELD] Concatenated 137340 field rows across all hexes
[ALL-HEX FIELD] SUCCESS — 17064 mobiles with combined field
Main scoring done  (587.1s)


/var/folders/hx/vjf09zlx6_s9jfgrk3vz2d7w0000gp/T/ipykernel_42862/2059315121.py:16: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->floating,key->block7_values] [items->Index(['parent_overlap'], dtype='str')]

  scored_df.to_hdf(cache_scored_h5, key="df", mode="w")


In [11]:
def _score_one_window(wd, df_train_full, df_test, df_poly, df_bound):
    """Score a single temporal window. Returns (wd, field_series, sources_series) or None."""
    ref_date = pd.Timestamp(config.TRAIN_END_DATE)
    cutoff = ref_date - pd.Timedelta(days=wd)
    df_tw = df_train_full.loc[df_train_full["decision_time"] >= cutoff]
    n_src = len(df_tw)
    if n_src < 50:
        return wd, None, None

    try:
        sw = process(
            df_tw.copy(), df_test, df_poly, df_bound,
            lambda_decay=config.LAMBDA_DECAY,
            max_radius_m=int(config.MIN_DIST_CUTOFF_M),
        )
        if sw is None or sw.empty:
            return wd, None, None

        out = sw[["mobile"]].copy()
        out[f"predicted_field_hex_{wd}d"] = sw.get("predicted_field_hex", np.nan)
        out[f"total_sources_field_{wd}d"] = sw.get("total_sources_all_hexes", 0)
        return wd, out, n_src
    except Exception as e:
        print(f"[TEMPORAL {wd}d] ERROR: {e}")
        return wd, None, None


In [12]:
t0 = time.perf_counter()
print("--- TEMPORAL FIELD SCORING (parallel) ---")

temporal_results = {}
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {
        pool.submit(
            _score_one_window, wd,
            df_train_full, df_test, df_poly, df_bound
        ): wd
        for wd in config.TEMPORAL_WINDOWS
    }
    for fut in as_completed(futures):
        wd, frame, n = fut.result()
        if frame is not None:
            temporal_results[wd] = frame
            nv = frame[f"predicted_field_hex_{wd}d"].notna().sum()
            print(f"  [{wd}d] {n} sources -> {nv}/{len(scored_df)} leads")
        else:
            print(f"  [{wd}d] skipped (too few sources or error)")
            scored_df[f"predicted_field_hex_{wd}d"] = np.nan
            scored_df[f"total_sources_field_{wd}d"] = 0


--- TEMPORAL FIELD SCORING (parallel) ---


  6%|▌         | 979/17572 [00:04<01:12, 229.61it/s]

  8%|▊         | 1465/17572 [00:07<01:26, 186.83it/s]

 11%|█         | 1955/17572 [00:09<00:52, 299.25it/s]

 26%|██▌       | 4587/17572 [00:19<00:47, 272.33it/s]

 27%|██▋       | 4671/17572 [00:20<01:06, 194.17it/s]

 28%|██▊       | 4982/17572 [00:21<00:44, 283.89it/s]

 44%|████▍     | 7692/17572 [00:32<00:39, 252.85it/s]

 68%|██████▊   | 12002/17572 [00:49<00:17, 321.35it/s]

 70%|███████   | 12372/17572 [00:50<00:15, 339.61it/s]

 71%|███████   | 12471/17572 [00:51<00:16, 309.09it/s]

 78%|███████▊  | 13785/17572 [00:55<00:14, 261.95it/s]

 92%|█████████▏| 16175/17572 [01:04<00:04, 281.41it/s]

 93%|█████████▎| 16374/17572 [01:05<00:04, 257.67it/s]

 99%|█████████▊| 17311/17572 [01:09<00:00, 276.32it/s]

100%|██████████| 17572/17572 [01:10<00:00, 250.66it/s]




[HEX CONSENSUS] Temporal windows available: [30, 60, 365]


[HOP FEATURES] Computing 3-hop neighbor SE aggregates (+ temporal)...





























100%|██████████| 17572/17572 [01:29<00:00, 195.94it/s]





...

[ALL-HEX FIELD] Concatenated 137340 field rows across all hexes
[ALL-HEX FIELD] SUCCESS — 14855 mobiles with combined field


  [30d] 53573 sources -> 14861/17578 leads


[ALL-HEX FIELD] Concatenated 137340 field rows across all hexes
[ALL-HEX FIELD] SUCCESS — 15934 mobiles with combined field


  [60d] 110578 sources -> 15940/17578 leads


[ALL-HEX FIELD] Concatenated 137340 field rows across all hexes
[ALL-HEX FIELD] SUCCESS — 17064 mobiles with combined field
  [365d] 721722 sources -> 17070/17578 leads


In [13]:
# Single merge pass: concat all temporal frames, then merge once
if temporal_results:
    # Each frame has 'mobile' + 2 cols; merge on mobile in one shot
    merged_temporal = None
    for wd in sorted(temporal_results):
        fr = temporal_results[wd].set_index("mobile")
        if merged_temporal is None:
            merged_temporal = fr
        else:
            merged_temporal = merged_temporal.join(fr, how="outer")

    scored_df = scored_df.merge(
        merged_temporal, on="mobile", how="left"
    )

# Derived: field momentum
w0, wn = config.TEMPORAL_WINDOWS[0], config.TEMPORAL_WINDOWS[-1]
col0 = f"predicted_field_hex_{w0}d"
coln = f"predicted_field_hex_{wn}d"
scored_df["field_momentum"] = (
    scored_df.get(col0, np.nan) - scored_df.get(coln, np.nan)
)
print(f"Temporal scoring done  ({time.perf_counter()-t0:.1f}s)")


Temporal scoring done  (1063.2s)


In [14]:
t0 = time.perf_counter()
test_start = dt.date.fromisoformat(config.TEST_START_DATE)
g1_start = (test_start - dt.timedelta(days=config.G1_LOOKBACK_EXTRA_DAYS)).isoformat()
g1_end = config.TEST_END_DATE

print(f"G1 window: {g1_start} -> {g1_end}")
df_g1 = get_g1_distance(g1_start, g1_end)

if df_g1.empty:
    print("Warning: G1 empty — min_dist buckets will be NaN")
    df = scored_df.copy()
else:
    keep = ["mobile", "min_dist", "g1_is_bdo_lead", "serviceable", "zone_alias"]
    df = scored_df.merge(df_g1[keep], on="mobile", how="inner")

bins = [-np.inf, 20, 40, 60, 100, np.inf]
labels = ["0-20", "20-40", "40-60", "60-100", "100+"]
df["min_dist_bucket"] = pd.cut(
    df["min_dist"], bins=bins, labels=labels,
    right=True, include_lowest=True,
)
print(f"G1 merge done  ({time.perf_counter()-t0:.1f}s)")


G1 window: 2025-10-05 -> 2025-11-09
GETTING G1 DECLINES FROM 2025-10-05 TO 2025-11-09
[WiomData] snowflake_select_start: select mobile, created_at as decision_time, min_dist, CASE
        WHEN min_dist <= 5 THEN 'A. 0-5m'
G1 merge done  (5.3s)


In [15]:
# Indeterminate tagging
indet = (
    (df["parent_color"] == "lightgreen")
    & (df["parent_installs"] <= config.INDETERMINATE_INSTALLS_CUTOFF)
)
df["parent_color"] = np.where(indet, "indeterminate", df["parent_color"])
df["parent_color_super"] = np.where(
    df["parent_color"] == "indeterminate", "lightgreen", df["parent_color"]
)

# Gravity score (fully vectorized, no nested np.where)
field = df["predicted_field_hex"]
total = df["total"]
gs = np.full(len(df), 0, dtype=np.int8)
gs[field.isna()] = -99
strong = field > config.FIELD_THRESHOLD
gs[strong & (total > config.PARENT_TOTAL_THRESHOLD)] = 2
gs[strong & (total <= config.PARENT_TOTAL_THRESHOLD) & (gs != -99)] = 1
df["gravity_score"] = gs

cgs = np.where(
    df["contested_field"].isna(), -99,
    np.where(df["contested_field"] > config.CONTEST_FIELD_THRESHOLD, 1, 0),
)
df["comp_gravity_score"] = cgs


In [16]:
conditions = [
    (df["gravity_score"] == 2) & (df["comp_gravity_score"] == 1),
    (df["gravity_score"] == 2),
    (df["gravity_score"] == 1) & (df["comp_gravity_score"] == 1),
    (df["gravity_score"] == 1),
    (df["comp_gravity_score"] == 1),
]
choices = [
    "A. STRONG Comp+ STRONG Field",
    "B. STRONG Field",
    "C. STRONG Comp + WEAK Field",
    "D. WEAK FIELD",
    "E. STRONG Comp",
]
df["bk_class_score"] = np.select(conditions, choices, default="F. Bad Field")

lg = df["parent_color_super"] == "lightgreen"
og = df["parent_color_super"] == "orange"
bc_set1 = {"A. STRONG Comp+ STRONG Field", "B. STRONG Field",
           "C. STRONG Comp + WEAK Field", "E. STRONG Comp"}
bc_set2 = {"A. STRONG Comp+ STRONG Field", "B. STRONG Field"}

df["booking_classifier_new"] = (
    (lg & df["bk_class_score"].isin(bc_set1))
    | (og & df["bk_class_score"].isin(bc_set2))
).astype(np.int8)
df["final_serviceability"] = df["booking_classifier_new"]

df["is_sparse"] = (df["local_density"] < config.SPARSE_DENSITY_THRESHOLD).astype(np.int8)
df["is_deep"]   = (df["depth_score_point_hex"] > config.DEPTH_SCORE_THRESHOLD).astype(np.int8)


/var/folders/hx/vjf09zlx6_s9jfgrk3vz2d7w0000gp/T/ipykernel_42862/4249401477.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["is_sparse"] = (df["local_density"] < config.SPARSE_DENSITY_THRESHOLD).astype(np.int8)
/var/folders/hx/vjf09zlx6_s9jfgrk3vz2d7w0000gp/T/ipykernel_42862/4249401477.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["is_deep"]   = (df["depth_score_point_hex"] > config.DEPTH_SCORE_THRESHOLD).astype(np.int8)


In [17]:
df = compute_composite(df, df_ops_train)

if (not df_ops_train.empty
    and "nmbr_active_leads" in df_ops_train.columns
    and "expected_daily_slots" in df_ops_train.columns):

    exp = df_ops_train[["partner_id", "nmbr_active_leads",
                         "expected_daily_slots"]].copy()
    exp["exposure_ratio"] = (
        exp["nmbr_active_leads"]
        / exp["expected_daily_slots"].replace(0, np.nan)
    ).fillna(0)
    exp["e_concentrated"] = (
        exp["exposure_ratio"] > sc.EXPOSURE_CONCENTRATION_FACTOR
    ).astype(int)



--- R (PROMISE GOVERNOR) — COMPOSITE SCORING ---
  norm_se_30d: 14881/18082 leads with temporal SE norm
  norm_field_30d: 15365/18082 leads with temporal field norm
  spatial_shrunk_30d: 18082/18082 leads with temporal spatial shrunk
  norm_se_60d: 16143/18082 leads with temporal SE norm
  norm_field_60d: 16444/18082 leads with temporal field norm
  spatial_shrunk_60d: 18082/18082 leads with temporal spatial shrunk
  norm_se_365d: 17574/18082 leads with temporal SE norm
  norm_field_365d: 17574/18082 leads with temporal field norm
  spatial_shrunk_365d: 18082/18082 leads with temporal spatial shrunk
  norm_momentum: 14881/18082 leads
  norm_field_momentum: 15365/18082 leads


In [18]:
def evaluate_bucket(df, bucket_name):
    total = len(df)
    if total == 0:
        return {"Bucket": bucket_name, "Vol": 0, "Installs": 0, "SE": 0.0}
    installs = int(df["installed_decision"].sum())
    return {
        "Bucket": bucket_name,
        "Vol": total, "Installs": installs,
        "SE": round(installs / total, 4),
    }


In [19]:
print("--- SEPARATION ANALYSIS ---")
pcs = df["parent_color_super"]
results = [
    evaluate_bucket(df[pcs == "lightgreen"],  "Green Hex"),
    evaluate_bucket(df[pcs == "orange"],      "Orange Hex"),
    evaluate_bucket(df[pcs == "crimson"],     "Crimson Hex"),
    evaluate_bucket(df[(pcs == "lightgreen") & (df["parent_overlap"] == True)],  "Green+Contested"),
    evaluate_bucket(df[(pcs == "lightgreen") & (df["parent_overlap"] == False)], "Green+Solo"),
    evaluate_bucket(df[(pcs == "lightgreen") & (df["is_sparse"] == 0)], "Green+Dense"),
    evaluate_bucket(df[(pcs == "lightgreen") & (df["is_sparse"] == 1)], "Green+Sparse"),
]
res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))


--- SEPARATION ANALYSIS ---
         Bucket  Vol  Installs     SE
      Green Hex 3167      2086 0.6587
     Orange Hex 9304      5523 0.5936
    Crimson Hex 5103      2587 0.5070
Green+Contested 1700      1132 0.6659
     Green+Solo  713       469 0.6578
    Green+Dense 2759      1830 0.6633
   Green+Sparse  408       256 0.6275


In [20]:
print("\n--- TEMPORAL FEATURE SEPARATION ---")
for wd in config.TEMPORAL_WINDOWS:
    for col in [f"weighted_se_{wd}d", f"predicted_field_hex_{wd}d", f"norm_se_{wd}d"]:
        if col in df.columns:
            valid = df[col].dropna()
            if len(valid):
                p10, p90 = valid.quantile(0.1), valid.quantile(0.9)
                print(f"  {col:>35s} — p10={p10:.4f}  p90={p90:.4f}  "
                      f"gap={p90-p10:.4f}")



--- TEMPORAL FEATURE SEPARATION ---
                      weighted_se_30d — p10=0.0000  p90=1.0000  gap=1.0000
              predicted_field_hex_30d — p10=-1.2484  p90=0.8484  gap=2.0967
                          norm_se_30d — p10=0.0648  p90=0.8998  gap=0.8350
                      weighted_se_60d — p10=0.0370  p90=1.0000  gap=0.9630
              predicted_field_hex_60d — p10=-1.0872  p90=0.7302  gap=1.8174
                          norm_se_60d — p10=0.1001  p90=0.9000  gap=0.8000
                     weighted_se_365d — p10=0.0758  p90=0.7857  gap=0.7100
             predicted_field_hex_365d — p10=-0.5956  p90=0.2314  gap=0.8270
                         norm_se_365d — p10=0.1001  p90=0.8997  gap=0.7997


In [21]:
t0 = time.perf_counter()
df["boundary_dist_bucket"] = "INSIDE"
outside = df["nearest_boundary_dist_m"].notna()
bins_d  = [0, 20, 40, 60, 100, np.inf]
lbl_d   = ["0-20", "20-40", "40-60", "60-100", "100+"]
bcuts   = pd.cut(
    df.loc[outside, "nearest_boundary_dist_m"],
    bins=bins_d, labels=lbl_d, right=True, include_lowest=True,
)
df.loc[outside, "boundary_dist_bucket"] = bcuts.astype(str).values

# --- batch all groupby reports ---
agg_spec = dict(total=("mobile", "count"), installs=("installed_decision", "sum"))

report_specs = [
    (["min_dist_bucket", "final_serviceability", "boundary_dist_bucket"],
     "install_rate_by_min_dist_bucket_final_serviceability_distance_from_boundary.csv"),
    (["min_dist_bucket", "final_serviceability", "nmbr_boundaries_within_100m"],
     "install_rate_by_min_dist_bucket_final_serviceability_boundary_count.csv"),
    (["min_dist_bucket", "final_serviceability"],
     "install_rate_by_min_dist_bucket_and_final_serviceability.csv"),
    (["min_dist_bucket", "geometry_regime", "final_serviceability"],
     "install_rate_by_min_dist_bucket_and_geometry_regime.csv"),
]

for grp_cols, fname in report_specs:
    tmp = (
        df.groupby(grp_cols, dropna=False)
          .agg(**agg_spec)
          .reset_index()
    )
    tmp["install_rate"] = tmp["installs"] / tmp["total"]
    tmp.to_csv(os.path.join(REPORTS_DIR, fname), index=False)

print(f"Groupby reports written  ({time.perf_counter()-t0:.1f}s)")


Groupby reports written  (0.0s)


In [22]:
if RUN_SIMULATE:
    run_declines_simulation(df_train, df_poly, df_bound, g1_start, g1_end, REPORTS_DIR)


In [23]:
t0 = time.perf_counter()

# Config snapshot
config_snapshot = {f: getattr(config, f, None) for f in sc.H_SNAPSHOT_FIELDS}
df["config_snapshot"] = json.dumps(config_snapshot)

report_csv = os.path.join(REPORTS_DIR, "test_scores.csv")
report_h5  = os.path.join(REPORTS_DIR, "test_scored.h5")

df.to_csv(report_csv, index=False)
cat_cols = df.select_dtypes(include=["category"]).columns
if len(cat_cols):
    df[cat_cols] = df[cat_cols].astype(str)
df.to_hdf(report_h5, mode="w", key="df")

if not df_ops_train.empty:
    df_ops_train.to_csv(os.path.join(REPORTS_DIR, "partner_ops_train_vector.csv"), index=False)
if not df_ops_test.empty:
    df_ops_test.to_csv(os.path.join(REPORTS_DIR, "partner_ops_test_vector.csv"), index=False)

print(f"Saved: {report_csv}")
print(f"Saved: {report_h5}")
print(f"Save done  ({time.perf_counter()-t0:.1f}s)")


Saved: /Users/Rohanchoudhary/Desktop/projs/genie_stocks/reports/test_scores.csv
Saved: /Users/Rohanchoudhary/Desktop/projs/genie_stocks/reports/test_scored.h5
Save done  (1.4s)


/var/folders/hx/vjf09zlx6_s9jfgrk3vz2d7w0000gp/T/ipykernel_42862/2447959530.py:14: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->floating,key->block15_values] [items->Index(['parent_overlap'], dtype='str')]

  df.to_hdf(report_h5, mode="w", key="df")


In [24]:
temporal_cols = [c for c in df.columns
                 if any(f"_{wd}d" in c for wd in config.TEMPORAL_WINDOWS)]
print(f"\n[TEMPORAL] {len(temporal_cols)} columns in final output:")
for c in sorted(temporal_cols):
    nv = df[c].notna().sum() if df[c].dtype != object else "N/A"
    print(f"  {c}: {nv}/{len(df)}")



[TEMPORAL] 102 columns in final output:
  contested_installs_30d: 14973/18082
  contested_installs_365d: 14973/18082
  contested_installs_60d: 14973/18082
  contested_se_30d: 14320/18082
  contested_se_365d: 14972/18082
  contested_se_60d: 14776/18082
  contested_total_30d: 14973/18082
  contested_total_365d: 14973/18082
  contested_total_60d: 14973/18082
  declines_30d: 18082/18082
  declines_365d: 18082/18082
  declines_60d: 18082/18082
  dense_score_30d: 7520/18082
  dense_score_365d: 15531/18082
  dense_score_60d: 10737/18082
  gully_score_30d: 7520/18082
  gully_score_365d: 15531/18082
  gully_score_60d: 10737/18082
  hop1_se_30d_wmean: 17574/18082
  hop1_se_365d_wmean: 17574/18082
  hop1_se_60d_wmean: 17574/18082
  hop2_se_30d_wmean: 17574/18082
  hop2_se_365d_wmean: 17574/18082
  hop2_se_60d_wmean: 17574/18082
  hop3_se_30d_wmean: 17574/18082
  hop3_se_365d_wmean: 17574/18082
  hop3_se_60d_wmean: 17574/18082
  hull_area_30d: 7520/18082
  hull_area_365d: 15531/18082
  hull_area_